# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets and their associated fields and columns using the `@id` attribute.

This will help in identifying how to extract and reference the correct data for the next steps.

In [ ]:
# Show all record sets and their fields/columns by @id
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found. Please check your dataset structure.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.name} | @id: {rs.id}")
        fields = getattr(rs, 'fields', [])
        if fields:
            for field in fields:
                print(f"  Field: {getattr(field, 'name', '')} | @id: {getattr(field, 'id', '')} | Data type: {getattr(field, 'data_type', '')}")
        columns = getattr(rs, 'columns', [])
        if columns:
            for column in columns:
                print(f"  Column: {getattr(column, 'name', '')} | @id: {getattr(column, 'id', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*For this dataset, we will attempt to load all record sets detected. Adjust the record set IDs as needed if required for your analysis.*

In [ ]:
# Extract data from each record set
import warnings
warnings.filterwarnings('ignore')

record_set_ids = [rs.id for rs in (metadata.record_sets or [])]
dataframes = {}

print(f"Found {len(record_set_ids)} record sets.")

if not record_set_ids:
    print("No record sets available. Please check the schema or dataset structure.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f"  No records found for {record_set_id}.")

# Display a preview of the first DataFrame if available
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Showing head for record set: {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

*We'll demonstrate with the first available numeric field in the first record set.*

In [ ]:
# Pick a record set and a numeric field for basic EDA
if dataframes:
    # Use the first loaded record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Working with record set: {record_set_id}")

    # Try to find the first numeric field (int/float-like)
    numeric_fields = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
        else:
            # try coercing
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_fields.append(col)
            except Exception:
                continue
    if not numeric_fields:
        print("No numeric fields detected.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # Compute threshold for filtering (e.g., 10th percentile)
        threshold = df[numeric_field].quantile(0.10)
        print(f"Filtering records where {numeric_field} > {threshold:.3f}")
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records: {len(filtered_df)} out of {len(df)}")

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        possible_group_fields = [col for col in df.columns if col not in numeric_fields and df[col].nunique() < 20]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found for aggregation.")
else:
    print("No DataFrames loaded; cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Make visuals for the first DataFrame and numeric field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Find a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
    else:
        print('No numeric field found for visualization.')
else:
    print('No data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the Croissant dataset metadata, identified record sets, and loaded available records using their `@id`.
- Basic EDA and visualization are possible for numeric fields.
- For more advanced analysis, consult the dataset's data dictionary and Croissant schema for field descriptions.
- All data manipulations reference entities using their `@id`, promoting clarity and reproducibility.